# Aralin 08 - Multi-Agent na Disenyo ng Pattern


## Setup


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Bakit Multi-Agent Systems?

Ang mga totoong gawain tulad ng pagpaplano ng biyahe ay nangangailangan ng maraming uri ng kadalubhasaan — lohistika, lokal na kaalaman, pagba-budget, at iba pa. Ang isang solong ahente na sumusubok na hawakan ang lahat ay mabilis na nagiging mahirap kontrolin.

Nilulutas ng multi-agent systems ito sa pamamagitan ng **espesyalisasyon**: ang bawat ahente ay nakatuon sa isang larangan ng kadalubhasaan, na nagbubunga ng mas mataas na kalidad ng mga resulta kaysa sa isang generalista. Pinapabuti rin nila ang **scalability** — maaari kang magdagdag ng mga bagong ahente (hal., isang flight specialist, isang kritiko ng restawran) nang hindi nire-rewrite ang umiiral na workflow. Ang mga ahente ay nagkokompos ng magkakasama sa pamamagitan ng isang istrukturadong pipeline, nagpapasa ng konteksto mula isa patungo sa susunod.


## Paglikha ng Mga Espesyal na Ahente


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Paggawa ng Isang Sunud-sunod na Daloy ng Trabaho

Pinapayagan ka ng `WorkflowBuilder` na pagdugtungin ang mga ahente sa isang naka-direktang grap. Dito tayo gagawa ng isang simpleng dalawang-hating pipeline: ang **TravelPlanner** ang gagawa ng itinerary, pagkatapos ay ang **TravelConcierge** ang susuri at magpapahusay dito.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pagdaragdag ng Higit pang mga Ahente sa Daloy ng Trabaho

Isa sa pinakamalaking benepisyo ng multi-agent na pattern ay kung gaano ito kadaling palawakin. Sa ibaba, nagdagdag tayo ng isang **BudgetReviewer** na ahente na sumusuri sa plano laban sa badyet ng manlalakbay, nagmamarka ng mga bagay na maaaring magpataas ng gastos lampas sa limitasyon, at nagmumungkahi ng mga alternatibong makakatipid ng pera. Ang daloy ng trabaho ay ngayon nagpapatakbo ng tatlong ahente nang sunud-sunod:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

Sa leksyong ito natutunan mo kung paano:

1. **Lumikha ng mga espesyalisadong ahente** — bawat isa ay may nakatuong papel (pagpaplano, concierge, pagsusuri sa badyet).
2. **Ikabit ang mga ahente sa isang sunud-sunod na workflow** gamit ang `WorkflowBuilder` at `add_edge`.
3. **Isalitrato ang output** mula sa isang multi-agent pipeline, sinusubaybayan kung aling ahente ang nagsasalita.
4. **Palawakin ang workflow** sa pamamagitan ng pagdagdag ng mga bagong ahente sa kadena nang hindi binabago ang mga umiiral na.

Ang disenyo ng multi-agent ay nagpapanatiling simple ng bawat ahente habang nakakalikha ng mas mayaman, mas masusing mga resulta kaysa sa anumang isang ahente lamang ang maaaring makamit nang mag-isa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pahayag ng Pagsasanggalang**:
Ang dokumentong ito ay isinalin gamit ang AI translation service na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o di-tiyak na pagsasalin. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing pinagmulan. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasaling-tao. Hindi kami mananagot para sa anumang pagkakaintindihan o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
